***
# Homework 9: SQL and APIs

**Course:** STAT 606 - Computing in Data Science and Statistics SP24

**Name:** Shrivats Sudhir

**NetID:** ssudhir2

**Email:** ssudhir2@wisc.edu

**Collaborators:** Samuel Merten, Amy Merkelz

**Date:** April 8th, 2024
***

In [1]:
import os
import sqlite3
import requests

## 1.) Warmup: `sqlite3` (3 points, spent $\approx$ 5 minutes)

**Here is a table similar to the ones that we saw in the lecture slides, describing some information about the colleges in the West Division of the Big 10 conference.**

![](image.png)

**Use `sqlite3` to create a database with a single table (in addition to the standard metainformation tables) called `t_big10west` that recreates the table in the figure above. That is, `t_big10west` should have five columns `(ID, University, City, State, Founded)`, and seven rows corresponding to the seven universities in the table. Save the database in a file called `big10.db`, and include this file in your submission.**

In [127]:
data = [(101, 'University of Illinois', 'Urbana', 'Illinois', 1867),
        (202, 'University of Iowa', 'Iowa City', 'Iowa', 1847),
        (303, 'University of Minnesota', 'Minneapolis', 'Minnesota', 1851),
        (404, 'University of Nebraska', 'Lincoln', 'Nebraska', 1869),
        (505, 'Northwestern University', 'Evanston', 'Illinois', 1851),
        (606, 'Purdue University', 'West Lafayette', 'Indiana', 1869),
        (707 , 'University of Wisconsin', 'Madison', 'Wisconsin', 1849)]

# If big10.db already exists, we'll get yelled at when we
# try to create a DB file now, so delete it if it
# already exists.
UNIV_DB_FILE = 'big10.db'
if os.path.exists( UNIV_DB_FILE ):
    os.remove( UNIV_DB_FILE )

# Create a transaction
conn = sqlite3.connect( UNIV_DB_FILE )
# Create a cursor object
cursor = conn.cursor()

# Create a table
cursor.execute('''
                CREATE TABLE t_big10west ('ID', 
                                          'University', 
                                          'City', 
                                          'State', 
                                          'Founded') 
                ''')

# Insert data into the table
cursor.executemany('''
                    INSERT INTO t_big10west
                    VALUES (?, ?, ?, ?, ?) 
                    ''', data)

# Commit the transaction
conn.commit()

# Close the transaction
conn.close()

In [128]:
# Create a transaction
conn = sqlite3.connect( UNIV_DB_FILE )
# Create a cursor object
cursor = conn.cursor()

for row in cursor.execute(''' 
                          SELECT * FROM t_big10west 
                          '''):
        print(row)

# Close the transaction
conn.close()

(101, 'University of Illinois', 'Urbana', 'Illinois', 1867)
(202, 'University of Iowa', 'Iowa City', 'Iowa', 1847)
(303, 'University of Minnesota', 'Minneapolis', 'Minnesota', 1851)
(404, 'University of Nebraska', 'Lincoln', 'Nebraska', 1869)
(505, 'Northwestern University', 'Evanston', 'Illinois', 1851)
(606, 'Purdue University', 'West Lafayette', 'Indiana', 1869)
(707, 'University of Wisconsin', 'Madison', 'Wisconsin', 1849)


**Oops! There’s a typo in that table. The University of Wisconsin was founded in 1848, not 1849. Write a SQL command to correct the corresponding entry of the table, and save it in a string-valued variable called `big10_correction`. You do not need to run this command (but you probably should, to check that it’s correct, and if you do, and you change the entry in the table in `big10.db`, that’s okay)**

In [129]:
# Create a transaction
conn = sqlite3.connect( UNIV_DB_FILE )
# Create a cursor object
cursor = conn.cursor()

cursor.execute(''' 
               UPDATE t_big10west 
               SET Founded = 1848
               WHERE ID = 707 AND University = 'University of Wisconsin' AND City = 'Madison' AND State = 'Wisconsin'
               ''')

for row in cursor.execute('''
                          SELECT * FROM t_big10west
                          '''):
    print(row)

# Close the transaction
conn.close()

(101, 'University of Illinois', 'Urbana', 'Illinois', 1867)
(202, 'University of Iowa', 'Iowa City', 'Iowa', 1847)
(303, 'University of Minnesota', 'Minneapolis', 'Minnesota', 1851)
(404, 'University of Nebraska', 'Lincoln', 'Nebraska', 1869)
(505, 'Northwestern University', 'Evanston', 'Illinois', 1851)
(606, 'Purdue University', 'West Lafayette', 'Indiana', 1869)
(707, 'University of Wisconsin', 'Madison', 'Wisconsin', 1848)


## 2.) Relational Databases and SQL (7 points, spent $\approx$ 25 minutes)

**In this problem, you’ll interact with a toy SQL database using Python’s built-in sqlite3 package. Documentation can be found at** 
<h5 align="center"> https://docs.python.org/3/library/sqlite3.html </h5>

**For this problem, we’ll use a popular toy SQLite database, called `Chinook`, which represents a digital music collection. See the documentation at:**
<h5 align="center"> https://github.com/lerocha/chinook-database/blob/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite </h5>

**or a more detailed explanation. We’ll use the `.sqlite` file `Chinook_Sqlite.sqlite`, which you should download from the GitHub page above.** 

**Note: Don’t forget to save the file in the directory that you’re going to compress and hand in, and make sure that you use a relative path when referring to the file, so that when the grading script runs your code on one of our machines the file path will still work!**

**Load the database using the Python `sqlite3` package. How many tables are in the database? Save the answer in the variable `n_tables`.**

In [130]:
Chinook = 'Chinook_Sqlite.sqlite'
conn = sqlite3.connect( Chinook )
cursor = conn.cursor()

n_tables = 0
for table in cursor.execute('''
                            SELECT *
                            FROM sqlite_master
                            '''):
    if table[0] == 'table':
        n_tables += 1
print(f'There are {n_tables} tables in {Chinook}.')

conn.close()

There are 11 tables in Chinook_Sqlite.sqlite.


**What are the names of the tables in the database? Save the answer as a list of strings, `table_names`.** 

**Note: you should write Python `sqlite3` code to answer this; don’t just look up the answer in the documentation!**

In [131]:
Chinook = 'Chinook_Sqlite.sqlite'
conn = sqlite3.connect( Chinook )
cursor = conn.cursor()

table_names = []
for table in cursor.execute('''
                            SELECT *
                            FROM sqlite_master
                            WHERE type = 'table'
                            '''):
    table_names.append(table[1])
    
print(f'The table names in {Chinook} are:')
print(table_names)

conn.close()

The table names in Chinook_Sqlite.sqlite are:
['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


**Write a function `list_album_ids_by_letter` that takes as an argument a single character (i.e., a string of length one) and returns a list of the primary keys of all the albums whose titles start with that character.** 

**Your function should ignore case, so that the inputs `“a”` and `“A”` yield the same results.** 

**Include error checking that raises an appropriate error in the event that the input is of the wrong type or if it is not a single character.**

In [132]:
def get_column_names(table_name, cursor):
    
    cursor.execute(f'''
                   SELECT * 
                   FROM {table_name}
                   ''')

    cursor.fetchall()

    columns = []
    for col in cursor.description:
        columns.append(col[0])
    
    return columns

In [133]:
def list_album_ids_by_letter(char):

    if not isinstance(char, (str, )):
        raise TypeError(f'char ({char}) must be of type str.')
    
    if len(char) != 1:
        raise ValueError(f'char ({char}) must be a single character.')
    
    char = char.upper()
    
    Chinook = 'Chinook_Sqlite.sqlite'
    conn = sqlite3.connect( Chinook )
    cursor = conn.cursor()

    AlbumID_idx = get_column_names('Album', cursor).index('AlbumId')
    Title_idx = get_column_names('Album', cursor).index('Title')

    result = []
    for table in cursor.execute('''
                                SELECT *
                                FROM Album
                                '''):
        album_title = table[Title_idx]
        if album_title[0] == char:
            result.append(table[AlbumID_idx])
            
    conn.close()

    return result

print(list_album_ids_by_letter('a'))

[10, 14, 15, 24, 26, 29, 74, 75, 85, 89, 90, 94, 95, 96, 120, 139, 160, 167, 168, 169, 203, 224, 232, 233, 248, 254, 272, 273, 285, 296, 307, 319]


**Write a function `list_song_ids_by_album_letter` that takes as an argument a single character and returns a list of the primary keys of all the songs whose album names begin with that letter (again ignoring case).** 

**As in `list_album_ids_by_letter`, your function should ignore case and perform error checking as appropriate.**

**Hint: you’ll need a JOIN statement here. You can use the `cursor.description` attribute to find out about tables and the names of their columns.**

In [134]:
def list_song_ids_by_album_letter(char):

    if not isinstance(char, (str, )):
        raise TypeError(f'char ({char}) must be of type str.')
    
    if len(char) != 1:
        raise ValueError(f'char ({char}) must be a single character.')
    
    char = char.upper()
    album_ids = list_album_ids_by_letter(char)
    
    Chinook = 'Chinook_Sqlite.sqlite'
    conn = sqlite3.connect( Chinook )
    cursor = conn.cursor()

    TrackId_idx = get_column_names('Track', cursor).index('TrackId')
    AlbumId_idx = get_column_names('Track', cursor).index('AlbumId')
    
    result = []
    for table in cursor.execute('''
                                SELECT *
                                FROM Track
                                '''):
        if table[AlbumId_idx] in album_ids:
            result.append(table[TrackId_idx])

    conn.close()

    return result

print(list_song_ids_by_album_letter('a'))

[85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 923, 924, 925, 926, 927, 928, 929, 930, 931, 932, 933, 934, 935, 936, 937, 938, 939, 940, 941, 942, 943, 944, 945, 946, 947, 948, 1073, 1074, 1075, 1076, 1077, 1078, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1133, 1134, 1135, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1143, 1144, 1145, 1146, 1147, 1148, 1149, 1150, 1151, 1152, 1153, 1154, 1155, 1156, 1157, 1201, 1202, 1203, 1204, 1205, 1206, 1207, 1208, 1209, 1210, 1211, 1212, 1213, 1214, 1215, 1216, 1217, 1218, 1219, 1220, 1221, 1222, 1223, 1224, 1225, 1226, 1227, 1228, 1229, 1230, 1231, 1232, 1233, 1234, 1479, 1480, 148

**Write a function `total_cost_by_album_letter` that takes as an argument a single character and returns the total cost of buying all the songs whose album begins with that letter.**

**This cost should be based on the tracks’ unit prices, so that the cost of buying a set of tracks is simply the sum of the unit prices of all the tracks in the set.**

**Again your function should ignore case and perform appropriate error checking.**

In [135]:
def total_cost_by_album_letter(char):

    if not isinstance(char, (str, )):
        raise TypeError(f'char ({char}) must be of type str.')
    
    if len(char) != 1:
        raise ValueError(f'char ({char}) must be a single character.')
    
    char = char.upper()
    song_ids = list_song_ids_by_album_letter(char)

    Chinook = 'Chinook_Sqlite.sqlite'
    conn = sqlite3.connect( Chinook )
    cursor = conn.cursor()

    TrackId_idx = get_column_names('Track', cursor).index('TrackId')
    UnitPrice_idx = get_column_names('Track', cursor).index('UnitPrice')

    summation = 0
    for table in cursor.execute('''
                                SELECT *
                                FROM Track
                                '''):
        if table[TrackId_idx] in song_ids:
            summation += table[UnitPrice_idx]

    conn.close()

    return summation

print(total_cost_by_album_letter('a'))

366.3100000000019


## 3.) Warmup: interacting with the Yelp API (5 points, spent $\approx$ 15 minutes)

**In this problem, you’ll get some practice working with the Yelp API, which we already saw in lecture.**

**First, you need to obtain an API key in order to authenticate to the Yelp API. Follow the instructions at**
<h5 align="center"> https://docs.developer.yelp.com/docs/fusion-authentication </h5>

**under the section titled “Create an app on Yelp’s Developers site”. You may fill in whatever information you like in the app information.** 

**Note that you will need a Yelp account to create an app, which you need in order to obtain an API key. If you do not feel comfortable doing this, please let me know promptly by email.**

**Once you have filled out your information, you will be given a ClientID and an API Key. This ID and key come with an associated 300 free calls to the Yelp API to use in the month following the day you create your app.** 

**You should not need anywhere near these 300 API calls to test your code, but if you do run out of API calls, please let me know promptly.**

In [82]:
# Successfully obtained API key to authenticate Yelp API.
yelp_api_key = 'UqnvT2UCj3Jj_6DG62kXYvZWwAZU2xISaFddyUjB5XmHokDBIbZxJOjlcLZYntm2D4OO8b82yzvreyLm1vme7cEt-V2BaOoDsAD36NLQ3VyhI445LZv2zgcWVFAXZnYx'
yelp_client_id = 'LCPZKyKBQpu91swB9LRc1g'

**Write a function called `near_msc` that takes three arguments: a string, a non-negative integer and another string, in that order, representing a search string, a distance in meters and a Yelp API key, respectively.**

**`near_msc( s, d, key )` should return a list of strings, representing the Yelp aliases of all of the establishments matching the given search string s that are within d meters meters of the statistics department (1300 University Ave, Madison WI), using the given API key.**

**The distance argument should default to 1000. You may have the `key` argument default however you want. Note that we are including this optional `key` argument so that when it comes time to test your code, we can swap out your API key for that of the instructor or the grader.** 

**It will be most convenient for you to have this argument default to your API key, but be sure to change this behavior before submitting the assignment if you do not wish to share your API key with the instructor and grader (we will use our own keys to test your code, anyway, of course).**

**Your function should perform error checking to ensure that the arguments are of the right type, and you should raise an appropriate error in the event that the distance argument is negative.** 

**Hint: you have my permission to modify the code from the slides, which already essentially carries out this operation.** 

**Second hint: see the documentation at**
<h5 align="center"> https://docs.developer.yelp.com/reference/v3_business_search </h5>

In [4]:
def near_msc(s, d = 1000, key = yelp_api_key):

    if not isinstance(s, (str, )):
        raise TypeError(f's ({s}) must be of type str.')
    
    if not isinstance(d, (int, float, )):
        raise TypeError(f'd ({d}) must be of type int or float.')
    if d < 0:
        raise ValueError(f'd ({d}) must be non-negative.')
    
    if not isinstance(key, (str, )):
        raise TypeError(f'key ({key}) must be of type str.')
    
    url = 'https://api.yelp.com/v3/businesses/search'
    headers = {'Authorization': f'Bearer {key}'}
    url_params = {'term': f'{s}',
                  'radius': f'{d}',
                  'location': '1300 University Ave, Madison WI'}
    
    r = requests.get(url, headers = headers, params = url_params)
    return [res['alias'] for res in r.json()['businesses']]

In [79]:
near_msc('restaurant', 1000)

['sweet-home-wisconsin-madison',
 'butterbird-madison',
 'fabiolas-spaghetti-house-and-deli-madison',
 'camp-cantina-no-title',
 'chopsticks-no-title-3',
 'the-library-cafe-and-bar-madison',
 'qq-express-madison',
 'steenbocks-on-orchard-madison',
 'saigon-sandwich-madison-madison',
 'maries-soul-food-madison',
 'kosharie-madison',
 'mickies-dairy-bar-madison',
 'jordans-big-ten-pub-madison',
 'the-sett-madison',
 'stadium-take-out-madison',
 'aldos-cafe-madison',
 'nams-noodle-and-karaoke-bar-madison-2',
 'luckys-1313-brew-pub-madison-2',
 'babcock-hall-dairy-store-madison',
 'south-cantina-madison']

**Write a function called `best_near_msc` that has the same signature as `near_msc` (i.e., takes the same arguments and has the same default behavior) and returns a string representing the alias of the highest-rated establishment matching the given search string and within the given distance of the statistics department.** 

**If no businesses exist inside the given distance, your function should return `None`.** 

**Note that the ratings of the businesses are rounded to the nearest half star, so you will likely have ties, which you may break arbitrarily.** 

**Hint: it will be easiest to retrieve some search results and look at the attributes of the resulting JSON objects. You’re looking for an attribute that corresponds to a rating.**

In [80]:
def best_near_msc(s, d = 1000, key = yelp_api_key):
    
    if not isinstance(s, (str, )):
        raise TypeError(f's ({s}) must be of type str.')
    
    if not isinstance(d, (int, )):
        raise TypeError(f'd ({d}) must be of type int.')
    if d < 0:
        raise ValueError(f'd ({d}) must be non-negative.')
    
    if not isinstance(key, (str, )):
        raise TypeError(f'key ({key}) must be of type str.')
    
    url = 'https://api.yelp.com/v3/businesses/search'
    headers = {'Authorization': f'Bearer {key}'}
    url_params = {'term': f'{s}',
                  'radius': f'{d}',
                  'location': '1300 University Ave, Madison WI'}
    
    r = requests.get(url, headers = headers, params = url_params)
    temp = r.json()

    if len(temp['businesses']) == 0:
        return None
    
    minimum = 0
    index = 0

    for i, res in enumerate(temp['businesses']):

        if res['rating'] > minimum:
            alias = res['alias']
            minimum = res['rating']
            index = i
            
        elif res['rating'] == minimum:
            if res['review_count'] > temp['businesses'][index]['review_count']:
                alias = res['alias']
                minimum = res['rating']
                index = i

    return alias

In [81]:
best_near_msc('restaurant', 1000)

'stadium-take-out-madison'

## 4.) Tracking Asteroids with NASA’s NeoWs API (10 points, spent $\approx$ _ minutes)

**In this problem, you’ll get more practice working with APIs, this time using one maintained by NASA for retrieving information about near earth objects (NEOs), asteroids that pass close to Earth. The documentation is available at**

<h5 align="center"> https://api.nasa.gov/ </h5>

**(scroll down to the API titled *Asteroids NeoWs*).**

**First and foremost, you’ll need an API key for accessing the service. You can get one at**

<h5 align="center"> https://api.nasa.gov/ </h5>

**You’ll need to supply an email address, which can be either your Wisconsin email or a personal email address.**

In [83]:
# Successfully obtained Nasa API Key
nasa_api_key = 'BZeQmerI1U4TenI9frtwTWNsrPY1TNqyL3ZEhPC2'

**We’ll use the Asteroids NeoWs Feed to retrieve Near Earth Objects based on the date of their closest approach to Earth. This can be done using the Feed service. If you read the documentation, you’ll see that the Feed API is accessible at**

<h5 align="center"> https://api.nasa.gov/neo/rest/v1/feed </h5>

**and takes three URL parameters: `start_date`, `end_date` and `api_key`.** 

* **`start_date` and `end_date` specify the start and end of a date range, both formatted as `YYYY-MM-DD`.**

* **`api_key` specifies the API key that you requested previously.**

**Retrieve a JSON object from the NASA NeoWs Feed API for January 1st, 2015 (i.e., set `start_date` and `end_date` to be ’2015-01-01’). You’ll notice that the JSON object has three attributes:**

* **`element_count`: the number of near earth objects that had their nearest approach during the time spanned by `start_date` and `end_date`.**

* **`near_earth_objects`: a JSON object whose attributes are the dates (represented by strings of the form `YYYY-MM-DD`) in the time spanned by `start_date` and `end_date`. Each such date attribute has as its value an array of JSON objects, each of which represents a near Earth object.**

* **`links`: URLs pointing to the “current” day, and the days before and after**

**The JSON object for January 1st, 2015 should have `element_count` attribute equal to 14. That is, if your JSON object is stored in `neo_json`, evaluating `neo_json['element_count']` should be return 14.** 

**JSON objects representing the NEOs are stored in the array `neo_json['near_earth_objects']['2015-01-01']`**

**If you pick out one of the JSON objects in this array, it should have attributes that include strings like `'estimated_diameter'` and `'is_potentially_hazardous_asteroid'`**

**Extract the names of all of these attributes and store them in a Python list called `neo_attrs`.**

In [ ]:
url = 'https://api.nasa.gov/neo/rest/v1/feed'
url_params = {'start_date': f'{start_date}',
              'end_date': f'{end_date}',
              'api_key': nasa_api_key}